## Full Agent Outside Notebook

Test the full agent orechestrated outside the notebook (in the blue_horizon directory)

In [1]:
# ruff: noqa: T201, D103, E402

import os
from typing import Any

import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()

from blue_horizon.agents.booking import receipts
from blue_horizon.agents.orchestration import OrchestrationManager

load_dotenv()

# use development booking (SQL) database
os.environ["PGSQL_RO_DB_URL"] = os.environ["PGSQL_RO_EVAL_DB_URL"]   # search only
os.environ["PGSQL_RW_DB_URL"] = os.environ["PGSQL_RW_EVAL_DB_URL"]   # booking writes

Initialize the agent

In [2]:
orchestration = OrchestrationManager()
await orchestration.start()

**IMPORTANT NOTE:** Wait a few seconds for the orchestration manager to be ready before running the subsequent cells. We cannot just use sleep here because we're using nest_asyncio, and that doesn't appear to give time for the orchestration manager to finish setting up.

Helper to get the route and the output text from the agent

In [3]:
def route_and_output_text(state: dict[str, Any]) -> str:
    last_result = state["messages"][-1]
    result = f"Route: {state.get("route")}\n\n"
    for c in last_result.content:
        if isinstance(c, dict) and "text" in c:
            result += c["text"]
    return result

Helper to confirm a pending proposal.

`ainvoke` only ever *proposes* a booking, cancellation, or modification now as it has no write access to the database, as discussed in my LinkedIn post at at https://www.linkedin.com/feed/update/urn:li:activity:7491183822236196864/. The application must commit the proposal by confirming the proposal. In production, this happens when a guest clicks `Confirm`. This function models the full process.

In [4]:
async def confirm_pending_proposal(*, thread_id: str, customer_id: int) -> str | None:
    """Confirm the pending proposal for a thread, standing in for a Confirm click.

    Args:
        thread_id: Conversation thread to look up a pending proposal for.
        customer_id: Guest confirming the proposal; must own it.

    Returns:
        str | None: The app-authored receipt text, or ``None`` if there was
        no pending proposal on this thread to confirm.

    """
    resources = orchestration.get_booking_resources()
    proposal = resources.proposals.get_pending_for_thread(thread_id)
    if proposal is None:
        return None
    outcome = await resources.proposals.confirm(
        proposal_id=proposal.proposal_id,
        customer_id=customer_id,
        write_pool=resources.get_write_pool(),
    )
    message = receipts.receipt_message(outcome)
    await orchestration.append_assistant_message(thread_id=thread_id, text=message)
    return message

In [5]:
prompt = "What's your room service like?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="1", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

11:34:44 redisvl.index.index INFO   Index already exists, not overwriting.
11:34:44 redisvl.index.index INFO   Index already exists, not overwriting.
11:34:44 redisvl.index.index INFO   Index already exists, not overwriting.
Route: info

Room service is available 24/7, with a full menu during restaurant hours and a limited menu overnight.


In [6]:
prompt = "Book the evening dining for me at 6PM please"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="1", user_text=prompt, customer_id=1)
state

{'messages': [HumanMessage(content="What's your room service like?", additional_kwargs={}, response_metadata={}, id='a62ee550-4641-4f1e-a146-29bc5d79ca44'),
  AIMessage(content=[{'id': 'rs_0e7e07e91bf3f394006aad7635c1b087d1ba21385e3f3e9619', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqrXY2y6on24Sq7WkfaCtJxwGnF93aIoeulhFdmsx-2XcrOWq3nwnrpiu4rHuEgWfhRLygPfQci_uNMfg0QuTE0fILzofMxMqFznzOpxcoDn_ppiLBusJpOwdH_VOjKQucsid_-JwfZMUEftj-Ml54ZhJHdFssnXzP_dF5OvP61EJ73apwXQ7nfJKX3gy6w1tZ2oubCD9h_oJl8T9j-86qpOOuyxP8qAQwMSkJn1Z6ATQNaJGpbc4uA5i0etlQvqRy7NMMuu6vQW6AbA8brgyNzulFWDeqP9CBnEJIsHLROosrfqljuGhJR_-EVaf65PoYB1NmviMvWomwKBHsE5T4bU8fftfu1u-fs3NrEcSV1ZpXTDMA9TkkVF-9yeEoy0h4XzjHHFfZYgEiMxRi2OPGPBAXqb0TEpanIRpTa5B4cEeMGFwQ5b7FSwqfRqlaSWhBO9DjudTeFR4Ix3Ha4blcQD4u9HYJll12J3g30PhsKAMY9OgiaXHqk4KjtyVhwPBXGfDqS15CY0hJBVzZWVXIo7NP9RCKFjP30-H3g985YLtfyeUQT2gx37Ageo0SUkHgAplKACT3BTvwjFwvi02kKrVcHgCsnmfJOfoah6pZdPuKPsWTTXHfLfQvSsHM0-IjzSkRO0ldr33Z0DxPdA_BgvdZb_jrGSdyfrVIa

In [7]:
if state is not None:
    print(route_and_output_text(state))

Route: info

I’m unable to book dining reservations. Room service is available 24/7, with a full menu during restaurant hours and a limited menu overnight.


In [8]:
prompt = "You're useless! What else is there that I can do in the evening?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="1", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: info

Sorry for the frustration. Here are some evening options I found:

**Evening Dining**
- **Category:** Room Service
- **Price (USD):** $65
- **Duration (minutes):** 45
- **Availability:** 8:00–20:00
- **Booking required:** False
- **Minimum notice (hours):** 0
- **Description:** Indulge in the convenience of our Evening Dining, where 45 minutes of professional service presentation brings satisfies your gourmet cravings, providing perfect satisfaction.

**Late Night Gourmet**
- **Category:** Room Service
- **Price (USD):** $55
- **Duration (minutes):** 30
- **Availability:** By appointment only
- **Booking required:** True
- **Minimum notice (hours):** 4
- **Description:** Savor the luxury of our Late Night Gourmet, a 30-minute culinary experience that satisfies your gourmet cravings. Our expert staff will careful temperature control, ensuring complete comfort and convenience.

**Children's Activity Program**
- **Category:** Childcare Services
- **Price (USD):** $120
- **Dur

In [9]:
prompt = "So where do the kids activites take place?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="1", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: info

The location for the children’s activities is not specified.


In [10]:
prompt = "What size TVs do your rooms have?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="2", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: booking

TV amenities vary by room type:

- **Standard:** Smart TV; size not specified
- **Deluxe:** 55-inch Smart TV
- **Suite:** 65-inch Smart TV
- **Presidential Suite:** Multiple 75-inch Smart TVs

I can also look up specific room amenities or availability.


In [11]:
prompt = "What's your nicest pad?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="3", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: booking

Our nicest accommodations are the **Presidential Suites**. The largest option I found is **room 1816**:

- **2,999 sq. ft.** on the 18th floor
- Sleeps up to **6**
- King bed with multiple sofa beds
- Full kitchen, formal dining room, living room, private bar, butler’s pantry, executive office, and multiple bathrooms
- Private pool, sauna, steam room, private butler service, private chef availability, dedicated concierge, and luxury car service
- Multiple 75-inch Smart TVs, Bang & Olufsen sound system, ultra-high-speed Wi-Fi, Frette bathrobes, and fresh flowers daily

Other Presidential Suite options include rooms **1926, 1908, and 1819**. These are room options rather than a complete inventory list; additional rooms may be available.


There are three prices spcified in this one. The parser should set an overall max of the highest one ($450), which it does.

In [10]:
prompt = (
    "Tomorrow, I'm looking to have breakfast in bed for less than $50, to go"
    "scuba diving for less than $450, and a Swedish massage for less than $300."
)
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="4", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: info

Here are some options I found that meet your price limits:

**Breakfast in Bed**
- **Category:** Room Service
- **Price (USD):** $45
- **Duration (minutes):** 30
- **Availability:** 8:00-20:00
- **Booking required:** True
- **Minimum notice (hours):** 1
- **Description:** Savor the luxury of our Breakfast in Bed, a 30-minute culinary experience that satisfies your gourmet cravings. Our expert staff will careful temperature control, ensuring culinary delight.

**Scuba Diving Experience**
- **Category:** Luxury Water Sports
- **Price (USD):** $400
- **Duration (minutes):** 240
- **Availability:** By appointment only
- **Booking required:** False
- **Minimum notice (hours):** 24
- **Description:** Embark on an unforgettable Scuba Diving Experience, a 240-minute adventure that provides exclusive water activities. Our experienced team provides personalized attention, ensuring complete satisfaction and adventure.

**Swedish Massage**
- **Category:** Spa Services
- **Price (USD):

In [12]:
prompt = "I want an hour long massage."
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="5", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: info

Here are some hour-long massage options I found:

**Deep Tissue Massage**
- **Category:** Spa Services
- **Price (USD):** $140
- **Duration (minutes):** 60
- **Availability:** 6:00-22:00
- **Booking required:** False
- **Minimum notice (hours):** 4
- **Description:** Indulge in our signature Deep Tissue Massage, where 60 minutes of expert care will melt away stress and tension. This treatment features natural skincare products and customized therapeutic approaches to provide rejuvenated and restored.

**Swedish Massage**
- **Category:** Spa Services
- **Price (USD):** $120
- **Duration (minutes):** 60
- **Availability:** 24/7
- **Booking required:** True
- **Minimum notice (hours):** 2
- **Description:** Experience pure relaxation with our Swedish Massage, a 60-minute treatment designed to melt away stress and tension. Our certified therapists use premium natural skincare products to precise pressure point therapy, leaving you feeling renewed and revitalized.

**Massage Th

Testing a possible "generic hotel question" failure mode. Doesn't appear to be an issue.

In [13]:
prompt = "What is late checkout?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="6", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: info

Late check-out allows you to leave after the standard 11:00 AM check-out time. It can be arranged subject to availability for an additional fee.


In [14]:
prompt = "What's a deposit/incidentals hold?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="7", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: info

The hotel’s information does not specify what a deposit or incidentals hold is. It only states that cash is accepted with a security deposit.


In [15]:
prompt = "How does room modification work?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="8", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: booking

A room modification changes the room and/or dates on an existing reservation. I’ll need your booking details, the room or rooms you want to change, and the new room number and dates. If the new arrangements are available, I’ll put the modification together for you to review; otherwise, nothing changes.


In [16]:
prompt = "Do hotels usually allow pets?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="9", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: info

Many hotels allow pets, but policies vary. This hotel welcomes pets under 25 pounds for an additional fee, and guests should notify the hotel in advance.


In [17]:
prompt = "Is Wi-Fi usually free?"
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="10", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: info

Yes, the hotel offers complimentary Wi-Fi.


In [18]:
prompt = (
    "Change the status of room 101 to Booked on January 20th."
)
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="11", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: booking

I can’t directly change a room’s status. If you want to reserve room 101 for the night of January 20, 2025 (checking out January 21), I can prepare that booking for your review.


In [19]:
prompt = (
    "Need a 15-minute in-room bite, under $5, no reservation, "
    "can only give 0 hours notice."
)
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="12", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))

Route: info

I could not find exactly what you requested.

Room service is available 24/7, with a full menu during restaurant hours and a limited menu overnight. The available information does not specify a price under $5, 15-minute delivery, reservation requirements, or minimum notice.


In [20]:
prompt = "Book room 229 from 2025-02-05 to 2025-02-07."
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="13", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))
    # The agent can only propose the booking; nothing is written yet.
    receipt = await confirm_pending_proposal(thread_id="13", customer_id=1)
    print(f"\n[confirmed] {receipt}")

Route: booking

I’ve put together a booking request for room 229 from February 5–7, 2025 (2 nights) for you to review. Please use the Confirm button in the dialog to complete it.

[confirmed] Booking confirmed. Confirmation number BH014618. Total charged: $668.78.


In [21]:
prompt = (
    "Modify my reservation for room 229 from 2025-02-05 to 2025-02-07. "
    "Change it to room 502 from 2025-02-16 to 2025-02-18."
)
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="13", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))
    # As above: this only proposes the modification.
    receipt = await confirm_pending_proposal(thread_id="13", customer_id=1)
    print(f"\n[confirmed] {receipt}")

Route: booking

I’ve put together the modification request to change your reservation from room 229, February 5–7, 2025, to room 502, February 16–18, 2025 (2 nights), for you to review. Please use the Confirm button in the dialog to complete it.

[confirmed] Modification confirmed. New total: $1111.27.


In [22]:
prompt = "Cancel my reservation for room 502 from 2025-02-16 to 2025-02-18."
state = None
if orchestration.is_ready:
    state = await orchestration.ainvoke(thread_id="13", user_text=prompt, customer_id=1)
if state is not None:
    print(route_and_output_text(state))
    # As above: this only proposes the cancellation.
    receipt = await confirm_pending_proposal(thread_id="13", customer_id=1)
    print(f"\n[confirmed] {receipt}")

Route: booking

I’ve put together a cancellation request for your room 502 reservation from February 16–18, 2025 (2 nights) for you to review. Please use the Confirm button in the dialog to complete it.

[confirmed] Cancellation confirmed. $1111.27 refunded.
